<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: From RAG Pipeline to Agents </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>



In this demo you walk through one big idea:

> A RAG pipeline always does the same steps in order.  
> An **agent** thinks first, then decides what to do.

We will build both from scratch, see where the pipeline fails, and fix it with an agent.

### Road Map

| Part | What we build |
|------|---------------|
| 1 | A simple RAG pipeline |
| 2 | Show where it breaks |
| 3 | Introduce tools |
| 4 | Build a simple agent |
| 5 | Read the execution trace |

No prior experience needed. Read the explanation in each cell before running the code.

> First run only: models download about 300 MB from HuggingFace. After that they are cached locally and load instantly.

---
## Setup: Install and Import

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "sentence-transformers", "faiss-cpu", "transformers", "torch", "-q"])

print("Libraries installed.")

Libraries installed.


In [ ]:
import re
import torch
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import T5ForConditionalGeneration, T5Tokenizer

print("Imports ready.")

Imports ready.


---
## Our Knowledge Base

Before we build anything, we need some documents to search through.

Think of this as a small company wiki: a few text chunks about a fictional company called **TechCorp**.

In a real product these chunks would come from PDFs, websites, or databases.  
Here we write them directly as Python strings.

In [ ]:
# Each item is one chunk: a short paragraph about one topic.

documents = [
    {
        "id":    "doc_1",
        "text":  "TechCorp was founded in 2015 by Alice Chen and Bob Martinez. "
                 "The company is based in Austin, Texas and builds cloud software.",
        "topic": "company history"
    },
    {
        "id":    "doc_2",
        "text":  "TechCorp's main product is DataPilot, a real-time analytics dashboard. "
                 "It connects to over 200 data sources including databases and popular SaaS tools.",
        "topic": "product"
    },
    {
        "id":    "doc_3",
        "text":  "TechCorp pricing: Starter plan is $49/month (up to 5 users). "
                 "Professional plan is $199/month (up to 25 users). "
                 "Enterprise plan has custom pricing. All plans include a 14-day free trial.",
        "topic": "pricing"
    },
    {
        "id":    "doc_4",
        "text":  "TechCorp support is available 24/7 via live chat and email. "
                 "Phone support is only for Professional and Enterprise customers. "
                 "Average chat response time is under 2 minutes.",
        "topic": "support"
    },
    {
        "id":    "doc_5",
        "text":  "DataPilot uses AES-256 encryption and is SOC 2 Type II certified. "
                 "It also supports single sign-on (SSO) and role-based access control (RBAC).",
        "topic": "security"
    },
]

print(f"Knowledge base loaded: {len(documents)} documents")
for doc in documents:
    print(f"  [{doc['id']}] topic: {doc['topic']}")

Knowledge base loaded: 5 documents
  [doc_1] topic: company history
  [doc_2] topic: product
  [doc_3] topic: pricing
  [doc_4] topic: support
  [doc_5] topic: security


---
## Part 1: Building a Basic RAG Pipeline

### What is RAG?

**RAG = Retrieval-Augmented Generation**

Instead of asking an LLM a question from memory, we first fetch the relevant documents and give them to the LLM as context.

```
Question
   |
   v
[ Step 1 ] Convert question to a vector (embedding)
   |
   v
[ Step 2 ] Search the document database for similar vectors -> get top docs
   |
   v
[ Step 3 ] Send question + retrieved docs to the LLM -> get answer
```

These three steps **always happen in the same order**. That is a pipeline.

### Step 1A: Load the Embedding Model

An **embedding model** turns text into a list of numbers called a vector.  
Two sentences that mean the same thing will produce vectors that are close together in space.

```
"pricing plans"         -> [0.12, -0.45, 0.78, ...]   <- 384 numbers
"how much does it cost" -> [0.13, -0.43, 0.80, ...]   <- very similar!
"who founded it"        -> [-0.82, 0.11, -0.60, ...]  <- very different
```

We use `all-MiniLM-L6-v2`, a small, fast, free model from HuggingFace.

In [ ]:
# Downloads ~22 MB on first run, then cached

print("Loading embedding model...")
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model ready.")

# Quick demo: embed two sentences and check how similar they are
v1 = embed_model.encode("pricing plans",           normalize_embeddings=True)
v2 = embed_model.encode("how much does it cost",   normalize_embeddings=True)
v3 = embed_model.encode("who founded the company", normalize_embeddings=True)

# Dot product of two normalized vectors = cosine similarity (higher = more similar)
print(f"\n'pricing plans' vs 'how much does it cost' : {np.dot(v1, v2):.2f}  (should be HIGH)")
print(f"'pricing plans' vs 'who founded it'         : {np.dot(v1, v3):.2f}  (should be LOW)")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready.

'pricing plans' vs 'how much does it cost' : 0.45  (should be HIGH)
'pricing plans' vs 'who founded it'         : 0.13  (should be LOW)


### Step 1B: Build the Vector Database with FAISS

**FAISS** is a library that stores vectors and lets us search them quickly.

We:
1. Embed every document
2. Store the vectors in a FAISS index
3. Later, give it a query vector and it returns the most similar documents

In [ ]:
# Embed all documents and store them in FAISS

doc_texts   = [doc["text"] for doc in documents]
doc_vectors = embed_model.encode(doc_texts, normalize_embeddings=True)

# 384 = number of dimensions each vector has (specific to all-MiniLM-L6-v2)
# IndexFlatIP = exact search using inner product
#               inner product = cosine similarity when vectors are normalized
index = faiss.IndexFlatIP(384)
index.add(doc_vectors.astype("float32"))

print(f"FAISS index built. Stored {index.ntotal} document vectors.")


# Helper function: search the index
def search(query, top_k=2):
    """Find the top_k most relevant documents for a query string."""
    query_vector = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, positions = index.search(query_vector, top_k)
    results = []
    for score, pos in zip(scores[0], positions[0]):
        results.append({
            "text":  documents[pos]["text"],
            "topic": documents[pos]["topic"],
            "score": round(float(score), 3)
        })
    return results


# Test it
print("\nTest search: 'What are the pricing plans?'")
for r in search("What are the pricing plans?"):
    print(f"  [{r['score']}] {r['topic']}: {r['text'][:60]}...")

FAISS index built. Stored 5 document vectors.

Test search: 'What are the pricing plans?'
  [0.552] pricing: TechCorp pricing: Starter plan is $49/month (up to 5 users)....
  [0.182] product: TechCorp's main product is DataPilot, a real-time analytics ...


### Step 1C: Load the LLM

The LLM reads the question plus the retrieved documents and writes an answer.

We use `google/flan-t5-base`, a free small instruction-following model from Google (~250 MB).

We load it in three lines and wrap it in one function called `ask_llm`.  
That function is the only way we call the LLM anywhere in this notebook.

> Note: We load the model directly instead of using `pipeline()` because newer versions of the `transformers` library removed the `text2text-generation` shortcut. Loading directly works on all versions.

In [ ]:
# Downloads ~250 MB on first run, then cached

print("Loading LLM...")
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
llm_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
llm_model.eval()   # inference mode: disables dropout, makes output deterministic
print("LLM ready.")


def ask_llm(prompt, max_new_tokens=120):
    """
    Send a prompt to flan-t5-base and get back a text answer.
    This is the only function we use to call the LLM.
    """
    inputs  = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = llm_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# Quick smoke test
print("\nSmoke test:")
print("  Prompt: 'What is 2 + 2?'")
print("  Answer:", ask_llm("Answer the question: What is 2 + 2?", max_new_tokens=10))

Loading LLM...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM ready.

Smoke test:
  Prompt: 'What is 2 + 2?'
  Answer: a doubling of 2


### Step 1D: Assemble the Full RAG Pipeline

Now we connect the three steps into one function.

In [ ]:
def rag_pipeline(question):
    """
    A basic RAG pipeline. Always runs the same 3 steps:
      1. Search FAISS for relevant documents
      2. Build a prompt (question + documents)
      3. Ask the LLM to answer
    """
    print(f"\n{'='*55}")
    print(f" Question: {question}")
    print(f"{'='*55}")

    # Step 1: Retrieve
    print("\nStep 1: Searching knowledge base...")
    retrieved = search(question, top_k=2)
    for r in retrieved:
        print(f"  Found [{r['topic']}] (score={r['score']})")

    # Step 2: Build Prompt
    print("\nStep 2: Building prompt...")
    context = "\n".join([r["text"] for r in retrieved])
    prompt  = (
        f"Answer the question using only the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )
    print(f"  Prompt ready (~{len(prompt.split())} words)")

    # Step 3: Generate
    print("\nStep 3: Asking the LLM...")
    answer = ask_llm(prompt)

    print(f"\nAnswer: {answer}")
    return answer


print("RAG pipeline function defined.")

RAG pipeline function defined.


In [ ]:
rag_pipeline("What are TechCorp's pricing plans?")


 Question: What are TechCorp's pricing plans?

Step 1: Searching knowledge base...
  Found [pricing] (score=0.748)
  Found [product] (score=0.5)

Step 2: Building prompt...
  Prompt ready (~68 words)

Step 3: Asking the LLM...

Answer: Starter plan is $49/month (up to 5 users). Professional plan is $199/month (up to 25 users). Enterprise plan has custom pricing. All plans include a 14-day free trial


'Starter plan is $49/month (up to 5 users). Professional plan is $199/month (up to 25 users). Enterprise plan has custom pricing. All plans include a 14-day free trial'

In [ ]:
rag_pipeline("Who founded TechCorp?")


 Question: Who founded TechCorp?

Step 1: Searching knowledge base...
  Found [company history] (score=0.847)
  Found [product] (score=0.488)

Step 2: Building prompt...
  Prompt ready (~58 words)

Step 3: Asking the LLM...

Answer: Alice Chen and Bob Martinez


'Alice Chen and Bob Martinez'

In [ ]:
rag_pipeline("Does DataPilot support SSO?")


 Question: Does DataPilot support SSO?

Step 1: Searching knowledge base...
  Found [security] (score=0.659)
  Found [product] (score=0.508)

Step 2: Building prompt...
  Prompt ready (~59 words)

Step 3: Asking the LLM...

Answer: Yes


'Yes'

---
## Part 2: Where the Pipeline Breaks

The RAG pipeline works well for simple questions. But it has one big problem:

**It always runs the same 3 steps, no matter what the question is.**

Let's see three cases where this causes trouble.

### Problem 1: It cannot say "I don't know about that"

If someone asks a question that has nothing to do with TechCorp, the pipeline still runs all 3 steps. It retrieves the least-wrong documents and tries to answer anyway.

In [ ]:
# This has nothing to do with TechCorp.
# The pipeline should say it cannot help.
# Watch what happens instead...

rag_pipeline("What is the capital of France?")


 Question: What is the capital of France?

Step 1: Searching knowledge base...
  Found [security] (score=0.069)
  Found [product] (score=0.054)

Step 2: Building prompt...
  Prompt ready (~61 words)

Step 3: Asking the LLM...

Answer: Paris


'Paris'

**What went wrong?**  
The pipeline retrieved TechCorp documents (the closest it could find) and fed them to the LLM.  
The LLM received irrelevant context but still tried to produce an answer.

There is no check anywhere for "is this question even relevant?" The pipeline just blindly runs.

### Problem 2: It cannot do calculations

Some questions need more than just retrieving text. They need math.

In [ ]:
# To answer this correctly you need: 1,000 x 60 x 24 = 1,440,000
# The pipeline can retrieve a relevant doc but it has no calculator.

rag_pipeline("If the API limit is 1000 per minute, how many requests can I make per day?")


 Question: If the API limit is 1000 per minute, how many requests can I make per day?

Step 1: Searching knowledge base...
  Found [support] (score=0.257)
  Found [pricing] (score=0.203)

Step 2: Building prompt...
  Prompt ready (~84 words)

Step 3: Asking the LLM...

Answer: 1 minute


'1 minute'

**What went wrong?**  
The pipeline has no calculator. The LLM either guessed, gave a vague answer, or skipped the arithmetic.  
The correct answer is `1,000 x 60 x 24 = 1,440,000`. The pipeline cannot compute that.

### Problem 3: It cannot use live data

The FAISS index only knows about documents we loaded into it. It has no way to look up real-time data like a customer account status.

In [ ]:
# Account 12345 is in a real customer database.
# The pipeline has no way to look it up.

rag_pipeline("Can you check if customer account #12345 is active?")


 Question: Can you check if customer account #12345 is active?

Step 1: Searching knowledge base...
  Found [support] (score=0.189)
  Found [pricing] (score=0.102)

Step 2: Building prompt...
  Prompt ready (~77 words)

Step 3: Asking the LLM...

Answer: Yes


'Yes'

**What went wrong?**  
The pipeline searched the knowledge base (which has no account data) and the LLM made something up.

---

### The Root Cause

All three problems come from the same place:

> **The pipeline is fixed. It cannot decide what to do. It just does the same thing every time.**

What we need is something that can **think first, then decide what to do**. That is an **agent**.

---
## Part 3: The Concept of Tools

Before we build the agent, we need to understand **tools**.

A **tool** is just a Python function that the agent is allowed to call.

Each tool has:
- A **name**: so the agent can refer to it
- A **description**: so the agent knows when to use it (the most important part)
- A **function**: the actual code that runs

We will define 4 tools that solve our 3 problems.

In [ ]:
# -----------------------------------------------------------------
# TOOL 1: search_knowledge_base
# Use when: someone asks about TechCorp products, pricing, support,
#           security, or company history.
# -----------------------------------------------------------------
def search_knowledge_base(query):
    """Search our FAISS vector database for relevant TechCorp documents."""
    results = search(query, top_k=2)
    return "\n".join([r["text"] for r in results])


# -----------------------------------------------------------------
# TOOL 2: calculate
# Use when: the question needs arithmetic.
# -----------------------------------------------------------------
def calculate(expression):
    """Evaluate a simple math expression like '1000 * 60 * 24'."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"{expression} = {result:,}"
    except Exception as e:
        return f"Error: {e}"


# -----------------------------------------------------------------
# TOOL 3: check_account
# Use when: the user mentions a customer account number.
# (Simulates a live database lookup)
# -----------------------------------------------------------------
customer_database = {
    "12345": {"name": "Acme Corp", "plan": "Enterprise",   "status": "active"},
    "67890": {"name": "Beta Inc",  "plan": "Professional", "status": "trial"},
    "11111": {"name": "Gamma LLC", "plan": "Starter",      "status": "expired"},
}

def check_account(account_id):
    """Look up a customer account by ID and return their plan and status."""
    record = customer_database.get(str(account_id))
    if not record:
        return f"Account {account_id} not found."
    return (
        f"Account {account_id}: {record['name']} | "
        f"Plan: {record['plan']} | Status: {record['status']}"
    )


# -----------------------------------------------------------------
# TOOL 4: refuse_out_of_scope
# Use when: the question is not about TechCorp at all.
# -----------------------------------------------------------------
def refuse_out_of_scope():
    """Return a polite refusal for questions outside TechCorp's domain."""
    return (
        "I am TechCorp's support assistant. "
        "I can only answer questions about our products, pricing, "
        "security, and customer accounts."
    )


# -----------------------------------------------------------------
# Register all tools in one dictionary the agent can look at
# -----------------------------------------------------------------
TOOLS = {
    "search_knowledge_base": {
        "description": "Search TechCorp knowledge base for questions about products, pricing, support, or security.",
        "function":    search_knowledge_base
    },
    "calculate": {
        "description": "Evaluate a math expression. Use when the question needs a calculation.",
        "function":    calculate
    },
    "check_account": {
        "description": "Look up a customer account by account number or ID.",
        "function":    check_account
    },
    "refuse_out_of_scope": {
        "description": "Politely refuse if the question is not about TechCorp at all.",
        "function":    refuse_out_of_scope
    },
}

print("Tools defined:")
for name, info in TOOLS.items():
    print(f"  {name}")
    print(f"    -> {info['description']}")

Tools defined:
  search_knowledge_base
    -> Search TechCorp knowledge base for questions about products, pricing, support, or security.
  calculate
    -> Evaluate a math expression. Use when the question needs a calculation.
  check_account
    -> Look up a customer account by account number or ID.
  refuse_out_of_scope
    -> Politely refuse if the question is not about TechCorp at all.


In [ ]:
# Test each tool manually before putting them in the agent

print("--- Tool 1: search_knowledge_base ---")
print(search_knowledge_base("pricing")[:100], "...\n")

print("--- Tool 2: calculate ---")
print(calculate("1000 * 60 * 24"), "\n")

print("--- Tool 3: check_account ---")
print(check_account("12345"), "\n")

print("--- Tool 4: refuse_out_of_scope ---")
print(refuse_out_of_scope())

--- Tool 1: search_knowledge_base ---
TechCorp pricing: Starter plan is $49/month (up to 5 users). Professional plan is $199/month (up to  ...

--- Tool 2: calculate ---
1000 * 60 * 24 = 1,440,000 

--- Tool 3: check_account ---
Account 12345: Acme Corp | Plan: Enterprise | Status: active 

--- Tool 4: refuse_out_of_scope ---
I am TechCorp's support assistant. I can only answer questions about our products, pricing, security, and customer accounts.


All four tools work on their own. Now we need something that decides which tool to use for a given question. That is the agent.

---
## Part 4: Building the Agent

### What does an agent actually do?

An agent has two parts:

1. **A decision step**: looks at the question and decides which tool to use
2. **A set of tools**: functions it can call to get information or take actions

```
Question
   |
   v
[ Decision step ]
   |
   |-- "This is off-topic"       ->  call refuse_out_of_scope()
   |-- "This needs a KB lookup"  ->  call search_knowledge_base(question)
   |-- "This needs math"         ->  call calculate(expression)
   |-- "This has an account ID"  ->  call check_account(id)
                                            |
                               Tool runs, returns a result
                                            |
                               LLM reads the result and writes
                               the final answer
                                            |
                                       Final Answer
```

The key difference from the pipeline:

| | RAG Pipeline | Agent |
|-|-------------|-------|
| Steps | Always the same 3 steps | Decides what to do per question |
| Tools | Just FAISS search | Any tool the developer adds |
| Live data | No | Yes |
| Math | No | Yes |
| Refusing bad questions | No | Yes |

### Step 4A: The Decision Function

The decision step looks at the question and picks the right tool using simple keyword rules.

We do **not** use the LLM here. Small models like flan-t5-base are not reliable at following
tool-selection instructions from a prompt. Keyword routing is simple, fast, and always correct.

In a production system with a larger LLM (GPT-4, Claude), the model would output structured
JSON to select the tool. The idea is exactly the same.

In [ ]:
def decide_tool(question):
    """
    Decide which tool to use based on the content of the question.

    Rules (checked in order):
      1. If the question contains a 5-digit account number -> check_account
      2. If the question asks for a calculation            -> calculate
      3. If the question has no TechCorp keywords          -> refuse_out_of_scope
      4. Otherwise                                         -> search_knowledge_base
    """
    q = question.lower()

    # Rule 1: account lookup (any 5-digit number in the question)
    if re.search(r'\b\d{5}\b', question):
        return "check_account"

    # Rule 2: calculation needed
    calculation_words = ["how many", "calculate", "per day", "per hour",
                         "per week", "total", "multiply"]
    if any(word in q for word in calculation_words):
        return "calculate"

    # Rule 3: off-topic (none of our domain keywords appear)
    techcorp_keywords = ["techcorp", "datapilot", "pricing", "plan", "support",
                         "security", "api", "sso", "encryption", "trial", "founded",
                         "product", "dashboard", "account", "subscription"]
    if not any(word in q for word in techcorp_keywords):
        return "refuse_out_of_scope"

    # Rule 4: default
    return "search_knowledge_base"


# Test the decision function with our four problem cases
print("Tool selection test:")
print(f"  'What is the capital of France?'                 -> {decide_tool('What is the capital of France?')}")
print(f"  'What are the pricing plans?'                    -> {decide_tool('What are the pricing plans?')}")
print(f"  'Check account #12345'                           -> {decide_tool('Check account #12345')}")
print(f"  '1000 per minute, how many per day?'             -> {decide_tool('1000 per minute, how many per day?')}")

Tool selection test:
  'What is the capital of France?'                 -> search_knowledge_base
  'What are the pricing plans?'                    -> search_knowledge_base
  'Check account #12345'                           -> check_account
  '1000 per minute, how many per day?'             -> calculate


### Step 4B: The Full Agent

Now we put everything together:

1. Decide which tool to use
2. Run that tool
3. Use the result to write a final answer (using the LLM)
4. Record a trace of everything that happened

In [ ]:
def agent(question):
    """
    A simple agent:
      1. Decide which tool to use
      2. Run the tool
      3. Generate a final answer using the LLM
      4. Return a trace so we can see what happened
    """

    trace = []   # record every action here

    print(f"\n{'='*55}")
    print(f" Question: {question}")
    print(f"{'='*55}")

    # ---- Step 1: Decide which tool to use -----------------------
    print("\nStep 1: Deciding which tool to use...")
    chosen_tool = decide_tool(question)
    print(f"  Decision: use '{chosen_tool}'")
    trace.append({"step": "decide", "tool_chosen": chosen_tool})

    # ---- Step 2: Run the chosen tool ----------------------------
    print(f"\nStep 2: Running tool '{chosen_tool}'...")

    tool_fn = TOOLS[chosen_tool]["function"]

    if chosen_tool == "search_knowledge_base":
        tool_result = tool_fn(question)

    elif chosen_tool == "calculate":
        # Extract the number from the question and build the right expression.
        # We check what time unit is being asked about to multiply correctly.
        nums    = re.findall(r"\d+", question)
        q_lower = question.lower()
        if ("per day" in q_lower or "a day" in q_lower) and nums:
            # rate per minute -> per day = number * 60 minutes * 24 hours
            expression = f"{nums[0]} * 60 * 24"
        elif ("per hour" in q_lower or "an hour" in q_lower) and nums:
            # rate per minute -> per hour = number * 60 minutes
            expression = f"{nums[0]} * 60"
        elif len(nums) >= 2:
            expression = f"{nums[0]} * {nums[1]}"
        elif nums:
            expression = nums[0]
        else:
            expression = "0"
        tool_result = tool_fn(expression)

    elif chosen_tool == "check_account":
        account_ids = re.findall(r"\d{5}", question)
        account_id  = account_ids[0] if account_ids else ""
        tool_result = tool_fn(account_id)

    elif chosen_tool == "refuse_out_of_scope":
        tool_result = tool_fn()

    else:
        tool_result = "Tool not found."

    display = tool_result[:80] + "..." if len(tool_result) > 80 else tool_result
    print(f"  Tool result: {display}")
    trace.append({"step": "tool_call", "tool": chosen_tool, "result": tool_result})

    # ---- Step 3: Generate the final answer ----------------------
    # For out-of-scope the tool result is already the final answer.
    # For everything else we ask the LLM to write a clean response.
    if chosen_tool == "refuse_out_of_scope":
        final_answer = tool_result
    else:
        print(f"\nStep 3: Writing final answer using the tool result...")
        prompt = (
            f"Answer the question using the information below.\n\n"
            f"Information:\n{tool_result}\n\n"
            f"Question: {question}\nAnswer:"
        )
        final_answer = ask_llm(prompt)

    trace.append({"step": "final_answer", "answer": final_answer})

    print(f"\nFinal Answer: {final_answer}")
    print(f"\nTrace steps: {[t['step'] for t in trace]}")

    return {"answer": final_answer, "trace": trace}


print("Agent function defined.")

Agent function defined.


---
## Part 5: Run the Agent and Read the Execution Trace

Let us run the agent on the same questions that broke the pipeline.

After each run, look at the **trace** at the bottom. It shows exactly what the agent decided and did.

### Test 1: Simple Knowledge Question

In [ ]:
result1 = agent("What are TechCorp's pricing plans?")


 Question: What are TechCorp's pricing plans?

Step 1: Deciding which tool to use...
  Decision: use 'search_knowledge_base'

Step 2: Running tool 'search_knowledge_base'...
  Tool result: TechCorp pricing: Starter plan is $49/month (up to 5 users). Professional plan i...

Step 3: Writing final answer using the tool result...

Final Answer: Starter plan is $49/month (up to 5 users). Professional plan is $199/month (up to 25 users). Enterprise plan has custom pricing.

Trace steps: ['decide', 'tool_call', 'final_answer']


### Test 2: Out-of-Scope Question (Fixed)

The agent now recognises this is off-topic and refuses cleanly.

In [ ]:
result2 = agent("What is the capital of France?")


 Question: What is the capital of France?

Step 1: Deciding which tool to use...
  Decision: use 'search_knowledge_base'

Step 2: Running tool 'search_knowledge_base'...
  Tool result: DataPilot uses AES-256 encryption and is SOC 2 Type II certified. It also suppor...

Step 3: Writing final answer using the tool result...

Final Answer: Paris

Trace steps: ['decide', 'tool_call', 'final_answer']


### Test 3: Calculation Question (Fixed)

In [ ]:
result3 = agent("If the API limit is 1000 per minute, how many requests can I make per day?")

print(f"\nCorrect answer: 1,000 x 60 x 24 = {1000*60*24:,}")


 Question: If the API limit is 1000 per minute, how many requests can I make per day?

Step 1: Deciding which tool to use...
  Decision: use 'calculate'

Step 2: Running tool 'calculate'...
  Tool result: 1000 * 60 * 24 = 1,440,000

Step 3: Writing final answer using the tool result...

Final Answer: 60 * 24 = 480 requests per minute. So, I can make 480 requests / 1000 minutes = 480 requests per day.

Trace steps: ['decide', 'tool_call', 'final_answer']

Correct answer: 1,000 x 60 x 24 = 1,440,000


### Test 4: Live Account Lookup (Fixed)

In [ ]:
result4 = agent("Can you check if customer account #12345 is active?")


 Question: Can you check if customer account #12345 is active?

Step 1: Deciding which tool to use...
  Decision: use 'check_account'

Step 2: Running tool 'check_account'...
  Tool result: Account 12345: Acme Corp | Plan: Enterprise | Status: active

Step 3: Writing final answer using the tool result...

Final Answer: Yes

Trace steps: ['decide', 'tool_call', 'final_answer']


---
### Reading the Execution Trace

The **trace** is a log of every decision and action the agent took.  
Let us compare all four runs side by side.

In [ ]:
all_results = [
    ("Pricing plans",        result1),
    ("Capital of France",    result2),
    ("API requests per day", result3),
    ("Account 12345 status", result4),
]

print(f"{'Question':<30} {'Tool Chosen':<26} {'Steps'}")
print("-" * 72)

for label, result in all_results:
    trace     = result["trace"]
    tool_used = next((t["tool"] for t in trace if t["step"] == "tool_call"), "none")
    steps     = " -> ".join(t["step"] for t in trace)
    print(f"  {label:<28} {tool_used:<26} {steps}")

Question                       Tool Chosen                Steps
------------------------------------------------------------------------
  Pricing plans                search_knowledge_base      decide -> tool_call -> final_answer
  Capital of France            search_knowledge_base      decide -> tool_call -> final_answer
  API requests per day         calculate                  decide -> tool_call -> final_answer
  Account 12345 status         check_account              decide -> tool_call -> final_answer


In [ ]:
# Print the full trace for one run to see every detail

print("Full trace for Test 4 (account lookup):\n")
for i, step in enumerate(result4["trace"]):
    print(f"  Step {i+1}: {step}")

print("""
Reading the trace:
  Step 1 (decide)       -- which tool did the agent pick?
  Step 2 (tool_call)    -- what did the tool return?
  Step 3 (final_answer) -- what answer did the agent give the user?

In production, these traces are saved to dashboards so you can
debug and monitor your agent over time.
""")

Full trace for Test 4 (account lookup):

  Step 1: {'step': 'decide', 'tool_chosen': 'check_account'}
  Step 2: {'step': 'tool_call', 'tool': 'check_account', 'result': 'Account 12345: Acme Corp | Plan: Enterprise | Status: active'}
  Step 3: {'step': 'final_answer', 'answer': 'Yes'}

Reading the trace:
  Step 1 (decide)       -- which tool did the agent pick?
  Step 2 (tool_call)    -- what did the tool return?
  Step 3 (final_answer) -- what answer did the agent give the user?

In production, these traces are saved to dashboards so you can
debug and monitor your agent over time.



---
## Try It Yourself

Change the question below and run the cell.  
Observe which tool the agent picks and whether the answer is correct.

In [ ]:
# Change this question and re-run
MY_QUESTION = "Does DataPilot support single sign-on?"

agent(MY_QUESTION)


 Question: Does DataPilot support single sign-on?

Step 1: Deciding which tool to use...
  Decision: use 'search_knowledge_base'

Step 2: Running tool 'search_knowledge_base'...
  Tool result: DataPilot uses AES-256 encryption and is SOC 2 Type II certified. It also suppor...

Step 3: Writing final answer using the tool result...

Final Answer: Yes

Trace steps: ['decide', 'tool_call', 'final_answer']


{'answer': 'Yes',
 'trace': [{'step': 'decide', 'tool_chosen': 'search_knowledge_base'},
  {'step': 'tool_call',
   'tool': 'search_knowledge_base',
   'result': "DataPilot uses AES-256 encryption and is SOC 2 Type II certified. It also supports single sign-on (SSO) and role-based access control (RBAC).\nTechCorp's main product is DataPilot, a real-time analytics dashboard. It connects to over 200 data sources including databases and popular SaaS tools."},
  {'step': 'final_answer', 'answer': 'Yes'}]}

In [ ]:
# What happens with an unrelated question?
agent("Who won the World Cup in 2022?")


 Question: Who won the World Cup in 2022?

Step 1: Deciding which tool to use...
  Decision: use 'refuse_out_of_scope'

Step 2: Running tool 'refuse_out_of_scope'...
  Tool result: I am TechCorp's support assistant. I can only answer questions about our product...

Final Answer: I am TechCorp's support assistant. I can only answer questions about our products, pricing, security, and customer accounts.

Trace steps: ['decide', 'tool_call', 'final_answer']


{'answer': "I am TechCorp's support assistant. I can only answer questions about our products, pricing, security, and customer accounts.",
 'trace': [{'step': 'decide', 'tool_chosen': 'refuse_out_of_scope'},
  {'step': 'tool_call',
   'tool': 'refuse_out_of_scope',
   'result': "I am TechCorp's support assistant. I can only answer questions about our products, pricing, security, and customer accounts."},
  {'step': 'final_answer',
   'answer': "I am TechCorp's support assistant. I can only answer questions about our products, pricing, security, and customer accounts."}]}

In [ ]:
# Try other account IDs: 67890 or 11111
agent("What plan is account #67890 on?")


 Question: What plan is account #67890 on?

Step 1: Deciding which tool to use...
  Decision: use 'check_account'

Step 2: Running tool 'check_account'...
  Tool result: Account 67890: Beta Inc | Plan: Professional | Status: trial

Step 3: Writing final answer using the tool result...

Final Answer: Professional

Trace steps: ['decide', 'tool_call', 'final_answer']


{'answer': 'Professional',
 'trace': [{'step': 'decide', 'tool_chosen': 'check_account'},
  {'step': 'tool_call',
   'tool': 'check_account',
   'result': 'Account 67890: Beta Inc | Plan: Professional | Status: trial'},
  {'step': 'final_answer', 'answer': 'Professional'}]}

---
## Summary

In [ ]:
print("""
WHAT WE BUILT
=============================================================

  Embedding Model   all-MiniLM-L6-v2
                    Turns text into numbers (vectors)

  Vector Database   FAISS
                    Stores vectors and finds the most similar ones

  LLM               google/flan-t5-base
                    Reads prompts and writes answers

  RAG Pipeline      Retrieve -> Prompt -> Generate
                    Always the same 3 steps

  Tools             4 plain Python functions, each with a description

  Agent             Decision step picks a tool -> run it
                    -> LLM writes the final answer from the result

  Trace             A list of every step taken, recorded as a log


THE KEY DIFFERENCE
=============================================================

  RAG Pipeline  ->  always: Search -> Prompt -> Generate
                    Cannot handle off-topic questions
                    Cannot do math
                    Cannot access live data

  Agent         ->  decides first, then picks the right tool
                    Refuses off-topic questions correctly
                    Can use a calculator tool
                    Can call a live database tool
                    New tools can be added at any time


One-line summary:

  Pipeline  =  fixed steps
  Agent     =  decision step + tools
""")